# Unit 10 solutions: model, stack, solve, and race a plan every tick

These reference solutions build the lesson's model predictive control from its pieces, check each piece where the answer is known (the linear model's own closed form, the tick-by-tick build, the least-squares optimum, the lesson's toy road), and then race it, always against the race's own line.
Each exercise's statement is repeated, followed by a prediction made before running anything, then the solution.
Every race is seeded and uses `MPC_BUDGET_MS = None`, so every race number below is reproducible, and executable asserts check each answer against an independent derivation.
The timings (Exercises 2 and 5) are this machine's, now: they are printed, and only their order is asserted.
Every number is a number about racekit's **carla-like car**: a model, not the truth.

In [ ]:
import math
import time
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10 = AGENTS / "unit-09.py", AGENTS / "unit-10.py"
workdir = Path(mkdtemp(prefix="roar-unit-10-sol-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
WHEELBASE = 2.875  # m, L (Unit 3)
MAX_STEER = math.radians(66.0)  # rad: the steering gain, measured (Unit 8), rounded as the agents round it
U_MAX = math.tan(MAX_STEER) / WHEELBASE  # 1/m: the tightest curvature full lock drives
A_THROTTLE = 6.0  # m/s^2 at full throttle (Unit 3's model)
WEIGHTS = (0.3, 20.0, 1000.0)  # (q_e, q_psi, r): agents/unit-10.py's MPC_Q_E, MPC_Q_PSI, MPC_R_DU
NO_TIMING = {"MPC_BUDGET_MS": None}  # every MPC race here: the horizon never depends on the machine's load

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
MEASURED_MU_G = {
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


def on_track(source, track_name, control, name, **constants):
    # A submission with this track's line and measured grip written in (Unit 9's idiom), plus any constants.
    return variant(source, name, LINE_CONTROL=tuple(float(value) for value in control),
                   MU_G=MEASURED_MU_G[track_name], **constants)


# Unit 9's line-relative clean rule.
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


# The lesson's reference along the line (section 2), read from an agent's own attributes.
def horizon_reference(agent, i, speed, n):
    # The profile speed (capped by full throttle) and the line's curvature at the distance the car will have
    # travelled after t + 1 ticks, t = 0 .. n - 1, starting from segment i.
    # (The agent also stops the walk after one lap; with n <= 80 ticks it never gets that far, so the two agree.)
    count = len(agent.profile)
    v_ref, k_ref = np.empty(n), np.empty(n)
    s_need, k, travelled = 0.0, i, 0.0
    v = max(speed, 1.0)
    for t in range(n):
        s_need += v * DT
        while travelled < s_need:
            travelled += agent.segment_length[k % count]
            k += 1
        v_ref[t] = max(min(agent.profile[k % count], speed + A_THROTTLE * DT * (t + 1)), 1.0)
        k_ref[t] = agent.curvature[k % count]
        v = v_ref[t]
    return v_ref, k_ref

## Exercise 1 - The error model, and where its linearisation fails

In the path frame, with a = Δt·v and the line's curvature κ, one Euler tick of the model **before** linearising is

e' = e + a sin ψ,  ψ' = ψ + a (u − κ cos ψ / (1 − κ e)),

and one tick of the **linearised** model is

e' = e + a ψ,  ψ' = ψ + a (u − κ).

Write:

- `linear_step(e, psi, u, a, kappa)` and `nonlinear_step(e, psi, u, a, kappa)`: each returns the tuple (e', ψ').

Use a corner of radius 25 m (κ = 0.04 1/m) at 15 m/s (a = Δt·15 = 0.75 m), and the command u = κ (the line's own curvature) throughout.

Define:

- `linear_step`, `nonlinear_step`: the functions;
- `one_tick_e_gap`: a dict from ψ₀ to |e'_linear − e'_nonlinear| after **one** tick from (e, ψ) = (0.0, ψ₀), for ψ₀ = 0.02, 0.1, 0.3 and 0.6 rad;
- `dropped_term`: κ·e for e = 0.5 m, the term the linearisation drops from 1 − κe;
- `blind_gaps`: a dict from ψ₀ to |e_linear − e_nonlinear| after **40** ticks (2 s) from (e, ψ) = (0.5, ψ₀), each model rolled from its own last state, for ψ₀ = 0.02 and 0.3;
- `fig`, `ax`: e against the tick for both models from (0.5, 0.3) over the 40 ticks, with a legend, axis labels and a title;
- `when_it_fails`: two sentences: when is the linear model a good model, and why does an MPC that plans again every tick need it to be good only for a short time near the line?

**Prediction.** From e = 0 the two models differ in e' only by a (ψ₀ − sin ψ₀), about a ψ₀³ / 6: a micrometre at 0.02 rad, a few centimetres at 0.6 rad.
The dropped term κe is 0.04 × 0.5 = 0.02, two percent of the 1 in 1 − κe.
Held blind for 40 ticks with u = κ, the linear model never turns (ψ' = ψ + a(κ − κ) = ψ), so it drifts by a ψ₀ every tick: 0.5 + 30 ψ₀, that is 1.1 m from ψ₀ = 0.02 and 9.5 m from ψ₀ = 0.3.
The nonlinear model, off the line on the left, sees the line turn faster than κ under it, so its heading error falls and it ends nearer the line; I expect a gap of tenths of a metre from the small heading error and metres from the large one.

In [ ]:
def linear_step(e, psi, u, a, kappa):
    return e + a * psi, psi + a * (u - kappa)


def nonlinear_step(e, psi, u, a, kappa):
    return e + a * math.sin(psi), psi + a * (u - kappa * math.cos(psi) / (1.0 - kappa * e))


KAPPA_CORNER = 0.04  # 1/m: a corner of radius 25 m
A_CORNER = DT * 15.0  # m: a = dt * v at 15 m/s


def roll_both(e, psi, ticks=40):
    # e after every tick for each model, each rolled from its own last state, with u = kappa throughout
    linear, nonlinear = (e, psi), (e, psi)
    linear_e, nonlinear_e = [], []
    for _ in range(ticks):
        linear = linear_step(*linear, KAPPA_CORNER, A_CORNER, KAPPA_CORNER)
        nonlinear = nonlinear_step(*nonlinear, KAPPA_CORNER, A_CORNER, KAPPA_CORNER)
        linear_e.append(linear[0])
        nonlinear_e.append(nonlinear[0])
    return np.array(linear_e), np.array(nonlinear_e)


one_tick_e_gap = {}
for psi0 in (0.02, 0.1, 0.3, 0.6):
    linear_e, nonlinear_e = roll_both(0.0, psi0, ticks=1)
    one_tick_e_gap[psi0] = abs(linear_e[0] - nonlinear_e[0])
dropped_term = KAPPA_CORNER * 0.5
blind_gaps = {}
for psi0 in (0.02, 0.3):
    linear_e, nonlinear_e = roll_both(0.5, psi0)
    blind_gaps[psi0] = abs(linear_e[-1] - nonlinear_e[-1])

linear_e, nonlinear_e = roll_both(0.5, 0.3)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(np.arange(1, 41), linear_e, label="linearised model")
ax.plot(np.arange(1, 41), nonlinear_e, label="before linearising")
ax.axhline(CLEAN_ERROR, color="grey", linestyle="--")
ax.set_xlabel("tick")
ax.set_ylabel("e (m, left +)")
ax.set_title("From (e, psi) = (0.5 m, 0.3 rad), u = kappa held blind for 2 s (R = 25 m, 15 m/s)")
ax.legend()
plt.show()
when_it_fails = (
    f"The linear model is good when the heading error is small and the car is near the line, for a short time: one "
    f"tick from psi = 0.02 rad the two models differ by {one_tick_e_gap[0.02]:.1e} m, but held blind for 2 s from "
    f"psi = 0.3 rad they end {blind_gaps[0.3]:.2f} m apart, more than the clean rule's 2 m. "
    "An MPC that plans again every tick only acts on the first command of each plan, from freshly measured errors "
    "that it keeps small, so the model only has to be right about the next few ticks near the line, where it is."
)

for psi0, gap in one_tick_e_gap.items():
    print(f"psi0 = {psi0:4.2f} rad: one tick, |e_linear - e_nonlinear| = {gap:.2e} m")
print(f"dropped term kappa * e = {dropped_term:.3f}")
for psi0, gap in blind_gaps.items():
    print(f"psi0 = {psi0:4.2f} rad: after 40 ticks the gap in e is {gap:.3f} m")
print(f"from (0.5, 0.3): linear e = {linear_e[-1]:.2f} m, nonlinear e = {nonlinear_e[-1]:.2f} m")
print(when_it_fails)

# independent checks: from e = 0 the one-tick gap is a (psi0 - sin psi0) exactly, about a psi0^3 / 6;
# with u = kappa the linear model's psi never changes, so its e grows by a psi0 a tick
for psi0, gap in one_tick_e_gap.items():
    assert abs(gap - A_CORNER * (psi0 - math.sin(psi0))) < 1e-15
for psi0 in (0.02, 0.1):  # small angles: psi0 - sin psi0 is psi0^3 / 6 to within 1%
    assert abs(one_tick_e_gap[psi0] - A_CORNER * psi0 ** 3 / 6) <= 0.01 * one_tick_e_gap[psi0]
assert one_tick_e_gap[0.02] < 1e-5 < 0.02 < one_tick_e_gap[0.6]
assert abs(dropped_term - 0.02) < 1e-15
for psi0 in (0.02, 0.3):
    linear_end, _ = roll_both(0.5, psi0)
    assert abs(linear_end[-1] - (0.5 + 40 * A_CORNER * psi0)) < 1e-12
assert blind_gaps[0.02] < 1.0 and blind_gaps[0.3] > CLEAN_ERROR
assert nonlinear_e[-1] < linear_e[-1]

## Exercise 2 - Stack the horizon, two ways

Over m ticks the errors x = (e_1, ψ_1, e_2, ψ_2, …, e_m, ψ_m) of the linearised model are x = G u + h.

Write:

- `loop_build(e, psi, a, kappa)`: G (2m × m) and h (2m) built one tick at a time, as the lesson's §3 did: the state (e, ψ) and the effect of every command so far are both multiplied by A_t = [[1, a_t], [0, 1]] each tick, the state also gets (0, −a_t κ_t), and command t's column starts as (0, a_t);
- `stack_horizon(e, psi, a, kappa)`: the closed form, with S = `np.cumsum(a)`: the ψ rows of G are a_j for j ≤ t (`np.tril(np.tile(a, (m, 1)))`), the e rows a_j (S_t − S_j) for j ≤ t, h_ψ = ψ − G_ψ κ and h_e = e + S ψ − G_e κ, interleaved e, ψ, e, ψ, ….

Test them on m = 30, `a = 0.05 * (12.0 + 8.0 * np.sin(0.2 * np.arange(30)))`, `kappa = 0.03 * np.cos(0.15 * np.arange(30))`, from (e, ψ) = (0.6, −0.04).

Define:

- `loop_build`, `stack_horizon`: the functions;
- `a_test`, `kappa_test`: the two arrays above;
- `same_G`, `same_h`: `np.allclose` of the two builds' G and of their h, with `rtol=1e-12, atol=1e-12`;
- `u_test`: `0.01 * np.sin(0.3 * np.arange(30))`, and `rollout_gap`: the largest |entry| of (G u_test + h) minus the errors you get by rolling Exercise 1's `linear_step` forward 30 ticks with u_test;
- `causal`: `True` when every entry of G in rows 2t and 2t + 1 and column j > t is exactly 0;
- `loop_ms`, `closed_ms`: the mean time of one build of each (ms), from 200 builds timed with `time.perf_counter()`;
- `why_lower_triangular`: one sentence: what does a zero above the diagonal of G say about the commands and the errors?

**Prediction.** The two builds are the same model written two ways, so G and h should agree to rounding, and G u + h should be the tick-by-tick rollout for any u.
A command cannot move an error that comes before it acts, so G is zero above the diagonal: `causal` is `True`; and e_1 cannot depend on u_0 at all (u_0 turns ψ first, then e moves one tick later), so even entry (0, 0) is 0.
The loop does m small matrix products in Python; the closed form a handful of whole-array operations, so it should be several times faster.

In [ ]:
def loop_build(e, psi, a, kappa):
    m = len(a)
    G = np.zeros((2 * m, m))
    h = np.zeros(2 * m)
    state = np.array([e, psi])
    effect = np.zeros((2, m))  # column j: what u_j has done to the current state so far
    for t in range(m):
        A_t = np.array([[1.0, a[t]], [0.0, 1.0]])
        state = A_t @ state + np.array([0.0, -a[t] * kappa[t]])
        effect = A_t @ effect
        effect[:, t] = (0.0, a[t])
        G[2 * t:2 * t + 2] = effect
        h[2 * t:2 * t + 2] = state
    return G, h


def stack_horizon(e, psi, a, kappa):
    m = len(a)
    S = np.cumsum(a)
    G_psi = np.tril(np.tile(a, (m, 1)))  # row t, column j: a_j for j <= t
    G_e = np.tril(S[:, None] - S[None, :]) * a[None, :]  # a_j (S_t - S_j) for j <= t
    h_psi = psi - G_psi @ kappa
    h_e = e + S * psi - G_e @ kappa
    G = np.empty((2 * m, m))
    G[0::2], G[1::2] = G_e, G_psi
    h = np.empty(2 * m)
    h[0::2], h[1::2] = h_e, h_psi
    return G, h


a_test = 0.05 * (12.0 + 8.0 * np.sin(0.2 * np.arange(30)))
kappa_test = 0.03 * np.cos(0.15 * np.arange(30))
G_loop, h_loop = loop_build(0.6, -0.04, a_test, kappa_test)
G_test, h_test = stack_horizon(0.6, -0.04, a_test, kappa_test)
same_G = np.allclose(G_loop, G_test, rtol=1e-12, atol=1e-12)
same_h = np.allclose(h_loop, h_test, rtol=1e-12, atol=1e-12)


def rollout(e, psi, u, a, kappa):
    # the errors (e_1, psi_1, e_2, psi_2, ...) from rolling linear_step forward tick by tick
    errors = []
    for t in range(len(u)):
        e, psi = linear_step(e, psi, u[t], a[t], kappa[t])
        errors += [e, psi]
    return np.array(errors)


u_test = 0.01 * np.sin(0.3 * np.arange(30))
rollout_gap = float(np.max(np.abs(G_test @ u_test + h_test - rollout(0.6, -0.04, u_test, a_test, kappa_test))))
causal = True
for t in range(30):
    for j in range(t + 1, 30):
        if G_test[2 * t, j] != 0.0 or G_test[2 * t + 1, j] != 0.0:
            causal = False


def mean_build_ms(build, repeats=200):
    started = time.perf_counter()
    for _ in range(repeats):
        build(0.6, -0.04, a_test, kappa_test)
    return (time.perf_counter() - started) / repeats * 1000


loop_ms, closed_ms = mean_build_ms(loop_build), mean_build_ms(stack_horizon)
why_lower_triangular = (
    "A zero above the diagonal says that a command cannot change an error that comes before it acts: u_j moves only "
    "the errors after tick j, so the plan can never rewrite the past."
)

print(f"same G: {same_G}, same h: {same_h}; rollout gap {rollout_gap:.1e}; causal: {causal}")
print(f"loop build {loop_ms:.3f} ms, closed form {closed_ms:.3f} ms ({loop_ms / closed_ms:.1f} times faster)")
print(why_lower_triangular)

# independent checks: entries worked by hand from one tick of the model, and h is the rollout with u = 0
a0, a1 = a_test[0], a_test[1]
assert G_test[0, 0] == 0.0 and G_test[1, 0] == a0  # e_1 does not see u_0 yet; psi_1 turns by a_0
assert abs(G_test[2, 0] - a0 * a1) < 1e-15 and G_test[2, 1] == 0.0  # e_2: u_0's heading, carried for a_1
assert abs(h_test[0] - (0.6 + a0 * -0.04)) < 1e-15 and abs(h_test[1] - (-0.04 - a0 * kappa_test[0])) < 1e-15
assert np.max(np.abs(h_test - rollout(0.6, -0.04, np.zeros(30), a_test, kappa_test))) < 1e-12
assert same_G and same_h and rollout_gap < 1e-12 and causal
assert closed_ms < loop_ms

## Exercise 3 - The least-squares plan, and what the weights do

The horizon cost is J(u) = Σ_{t=1}^{m} (q_e e_t² + q_ψ ψ_t²) + r Σ_{t=0}^{m−1} (u_t − u_{t−1})², with u_{−1} the last command applied.

Write:

- `least_squares_plan(G, h, u_last, weights=WEIGHTS)`: the plan u minimising J, from **one** `np.linalg.lstsq` call on the stacked rows [W G; √r D] and right-hand side [−W h; √r d], where W's diagonal is (√q_e, √q_ψ) repeated m times (`np.tile`), D = `np.eye(m) - np.eye(m, k=-1)` and d = (u_last, 0, …, 0);
- `horizon_cost(e, psi, u, a, kappa, u_last, weights=WEIGHTS)`: J computed by rolling `linear_step` forward tick by tick (no matrices).

Use Exercise 2's test horizon (from (0.6, −0.04), `a_test`, `kappa_test`) and u_last = 0.

Define:

- `least_squares_plan`, `horizon_cost`: the functions;
- `plan`: the plan with `WEIGHTS`;
- `cost_of_plan`, `cost_of_line`: J of `plan`, and J of the plan u = κ (`kappa_test`: drive the line's curvature and never correct);
- `beats_nudges`: `True` when `cost_of_plan` is below J of `plan + 1e-4 * np.sin(k * np.arange(30))` for every k = 1 … 5;
- `largest_change`: a dict from r to the largest |u_t − u_{t−1}| of the plan (including u_0 − u_last), for weights (0.3, 20.0, r) with r = 10.0, 1000.0 and 100000.0;
- `final_error`: a dict from the same r to |e_30|, the planned cross-track error at the end of the horizon;
- `first_command`: `plan[0]` clipped to ±`U_MAX`, and `wheel_angle_deg`: the wheel angle that drives it, atan(L u), in degrees;
- `what_r_does`: two sentences: what does a larger r buy, and what does it cost?

**Prediction.** The least-squares plan is the exact minimum of J for the linear model, so J(plan) is below J of any other plan: below the line's own curvature (which never corrects the 0.6 m) and below every small nudge.
Rolling the model tick by tick and squaring by hand must give the same J as the stacked vector, since they are the same sum.
A larger r makes every change of command dearer, so the largest change should shrink as r grows; with changes that slow, the plan cannot bring e down within 30 ticks, so I expect the end error to grow with r, a lot at r = 100000.
The start is 0.6 m left of the line and nearly lined up, so the first command steers gently right of the line's curvature: a hundredth of 1/m or so, far inside full lock (0.78 1/m), a wheel angle of a degree or two.

In [ ]:
def least_squares_plan(G, h, u_last, weights=WEIGHTS):
    q_e, q_psi, r = weights
    m = G.shape[1]
    w = np.tile([math.sqrt(q_e), math.sqrt(q_psi)], m)
    D = np.eye(m) - np.eye(m, k=-1)  # (D u)_t = u_t - u_{t-1}, and (D u)_0 = u_0 ...
    d = np.zeros(m)
    d[0] = u_last  # ... so the first change row is u_0 - u_last
    rows = np.vstack([w[:, None] * G, math.sqrt(r) * D])
    rhs = np.concatenate([-w * h, math.sqrt(r) * d])
    return np.linalg.lstsq(rows, rhs, rcond=None)[0]


def horizon_cost(e, psi, u, a, kappa, u_last, weights=WEIGHTS):
    q_e, q_psi, r = weights
    total, previous = 0.0, u_last
    for t in range(len(u)):
        e, psi = linear_step(e, psi, u[t], a[t], kappa[t])
        total += q_e * e * e + q_psi * psi * psi + r * (u[t] - previous) ** 2
        previous = u[t]
    return total


plan = least_squares_plan(G_test, h_test, 0.0)
cost_of_plan = horizon_cost(0.6, -0.04, plan, a_test, kappa_test, 0.0)
cost_of_line = horizon_cost(0.6, -0.04, kappa_test, a_test, kappa_test, 0.0)
nudged_costs = [horizon_cost(0.6, -0.04, plan + 1e-4 * np.sin(k * np.arange(30)), a_test, kappa_test, 0.0)
                for k in range(1, 6)]
beats_nudges = cost_of_plan < min(nudged_costs)

largest_change, final_error = {}, {}
for r in (10.0, 1000.0, 100000.0):
    plan_r = least_squares_plan(G_test, h_test, 0.0, (0.3, 20.0, r))
    largest_change[r] = float(np.max(np.abs(np.diff(np.concatenate([[0.0], plan_r])))))
    final_error[r] = abs((G_test @ plan_r + h_test)[-2])  # the e row of the last tick
first_command = min(max(plan[0], -U_MAX), U_MAX)
wheel_angle_deg = math.degrees(math.atan(WHEELBASE * first_command))
what_r_does = (
    f"A larger r buys a smoother wheel: the largest change of command falls from {largest_change[10.0]:.4f} to "
    f"{largest_change[100000.0]:.4f} 1/m between r = 10 and r = 100000, so the car is not jerked (which on a car "
    "that slides is what keeps it from weaving). "
    f"It costs tracking: at r = 100000 the plan is too slow to correct, and still {final_error[100000.0]:.2f} m off "
    f"the line at the end of the horizon, against {final_error[1000.0]:.3f} m at r = 1000 (r = 10, "
    f"{final_error[10.0]:.3f} m, is no better at the end: it spends its cheap changes on the heading early)."
)

print(f"J(plan) = {cost_of_plan:.6f}; J(u = kappa) = {cost_of_line:.6f}; best nudged {min(nudged_costs):.6f}; "
      f"beats every nudge: {beats_nudges}")
for r in (10.0, 1000.0, 100000.0):
    print(f"r = {r:8.0f}: largest change {largest_change[r]:.5f} 1/m, planned |e_30| {final_error[r]:.4f} m")
print(f"first command {first_command:.5f} 1/m (full lock {U_MAX:.3f} 1/m): wheel angle {wheel_angle_deg:.3f} deg")
print(what_r_does)

# independent checks: J as the squared length of the stacked residual, the normal equations at the optimum
q_e, q_psi, r_du = WEIGHTS
x_plan = G_test @ plan + h_test
change = np.diff(np.concatenate([[0.0], plan]))
stacked_J = q_e * np.sum(x_plan[0::2] ** 2) + q_psi * np.sum(x_plan[1::2] ** 2) + r_du * np.sum(change ** 2)
assert abs(stacked_J - cost_of_plan) < 1e-10 * cost_of_plan
w_diag = np.tile([math.sqrt(q_e), math.sqrt(q_psi)], 30)
D_check = np.eye(30) - np.eye(30, k=-1)
gradient = (w_diag[:, None] * G_test).T @ (w_diag * x_plan) + r_du * D_check.T @ (D_check @ plan)
assert np.max(np.abs(gradient)) < 1e-9  # the gradient of J / 2 is zero at the plan
assert cost_of_plan < cost_of_line and beats_nudges
assert largest_change[10.0] > largest_change[1000.0] > largest_change[100000.0]
assert final_error[100000.0] > 5 * final_error[1000.0]
assert plan[0] < kappa_test[0] and abs(plan[0]) < 0.1 * U_MAX  # right of the line's curvature, far inside full lock
assert first_command == plan[0]  # so the clip leaves it alone
assert abs(wheel_angle_deg) < 3.0

## Exercise 4 - The lag, on a toy road

A toy road: straight, then from 30 m a 60 m arc of radius 40 m (κ = 1/40), then straight again.
A toy car drives it at a steady 18 m/s; its errors follow Exercise 1's `nonlinear_step`, and it obeys each command **`car_lag` ticks after it was sent** (the commands in flight wait in a queue, oldest first, all 0 at the start).
It starts at (e, ψ) = (0.8, 0.0).

Write:

- `toy_kappa(distance)`: the road's curvature at a distance along it;
- `mpc_tick(e, psi, a, kappa, queue, u_last, weights=WEIGHTS)`: one MPC tick, as the lesson's: roll `linear_step` through the queued commands (the first `len(queue)` ticks of a and kappa), stack the rest of the horizon with `stack_horizon`, solve with `least_squares_plan`, and return the first command clipped to ±`U_MAX`;
- `toy_drive(car_lag, model_lag, ticks=140, speed=18.0, horizon=40)`: the closed loop; every tick, with `distance` the distance driven so far,
  1. the controller's horizon: a_t = Δt·speed and κ_t = `toy_kappa(distance + speed * DT * (t + 1))`, t = 0 … horizon − 1;
  2. the controller's queue is the **last `model_lag`** commands it sent (oldest first; none when `model_lag` is 0), and u_last is the last command it sent;
  3. the car moves one tick with the command it obeys now (the oldest of the `car_lag` commands in flight; with `car_lag` 0, the command just sent) and κ = `toy_kappa(distance + speed * DT)`;
  4. `distance` grows by speed·Δt.

  It returns the array of e after each tick.

Define:

- `toy_kappa`, `mpc_tick`, `toy_drive`: the functions;
- `worst_error`: a dict from (car_lag, model_lag) to the largest |e| after the first 20 ticks, for (car_lag, model_lag) = (1, 1), (1, 0), (2, 2), (2, 1) and (2, 0);
- `fig`, `ax`: e against time (s) for car_lag = 2 and the three model lags, with a legend, axis labels and a title;
- `toy_versus_car`: two or three sentences: on the toy, how much does it cost the controller to forget some of the lag? The lesson's race with `MPC_LAG = 0` was not clean on the carla-like car: why can a kinematic toy at a steady speed not settle that question?

**Prediction.** With `car_lag = 1` this is the lesson's toy, so (1, 1) and (1, 0) should reproduce its numbers: about 0.13–0.14 m and 0.12 m at worst after the first second.
A two-tick lag gives the car one more tick of the wrong command at every change, so I expect the worst errors to grow a little with `car_lag = 2`.
The toy is kinematic, at a steady speed, measured perfectly, and the corrections are small and smooth (r = 1000), so a command one or two ticks late is nearly the same command: I expect forgetting the lag to cost the toy almost nothing, centimetres at most.

In [ ]:
def toy_kappa(distance):
    return 1.0 / 40.0 if 30.0 <= distance < 90.0 else 0.0


def mpc_tick(e, psi, a, kappa, queue, u_last, weights=WEIGHTS):
    for t in range(len(queue)):
        e, psi = linear_step(e, psi, queue[t], a[t], kappa[t])
    G, h = stack_horizon(e, psi, a[len(queue):], kappa[len(queue):])
    plan = least_squares_plan(G, h, u_last, weights)
    return min(max(plan[0], -U_MAX), U_MAX)


def toy_drive(car_lag, model_lag, ticks=140, speed=18.0, horizon=40):
    e, psi, distance = 0.8, 0.0, 0.0
    sent = [0.0] * max(car_lag, model_lag, 1)  # every command sent so far, oldest first (zeros before the start)
    errors = []
    for _ in range(ticks):
        a = np.zeros(horizon) + DT * speed
        kappa = np.array([toy_kappa(distance + speed * DT * (t + 1)) for t in range(horizon)])
        queue = sent[len(sent) - model_lag:] if model_lag > 0 else []
        sent.append(mpc_tick(e, psi, a, kappa, queue, sent[-1]))
        obeyed = sent[len(sent) - 1 - car_lag]  # the command sent car_lag ticks ago (car_lag 0: the one just sent)
        e, psi = nonlinear_step(e, psi, obeyed, DT * speed, toy_kappa(distance + speed * DT))
        distance += speed * DT
        errors.append(e)
    return np.array(errors)


drives = {}
worst_error = {}
for lags in ((1, 1), (1, 0), (2, 2), (2, 1), (2, 0)):
    drives[lags] = toy_drive(*lags)
    worst_error[lags] = float(np.max(np.abs(drives[lags][20:])))

fig, ax = plt.subplots(figsize=(11, 3.5))
for model_lag in (2, 1, 0):
    errors = drives[(2, model_lag)]
    ax.plot(DT * np.arange(1, len(errors) + 1), errors, label=f"car lag 2, model lag {model_lag}")
ax.axvline(30.0 / 18.0, color="grey", linewidth=0.8)
ax.axvline(90.0 / 18.0, color="grey", linewidth=0.8)
ax.set_xlabel("t (s); between the grey lines: the arc")
ax.set_ylabel("e (m)")
ax.set_title("The toy road with a two-tick lag: how much of it the model knows")
ax.legend()
plt.show()
drives_2 = [worst_error[(2, model_lag)] for model_lag in (2, 1, 0)]
toy_versus_car = (
    f"On the toy, forgetting the lag costs almost nothing: with a two-tick lag the worst error after the first second "
    f"is {worst_error[(2, 2)]:.3f} m knowing all of it, {worst_error[(2, 1)]:.3f} m knowing one tick and "
    f"{worst_error[(2, 0)]:.3f} m knowing none: within {100 * (max(drives_2) - min(drives_2)):.0f} cm of each other, "
    "and knowing all of it is not even the best. "
    "The toy is kinematic, at a steady speed, measured perfectly, so a command one or two ticks late is nearly the "
    "same gentle command and nothing grows from the difference; the carla-like car's tyres slip and its speed "
    "changes, and a late correction there can feed a slide that the next correction makes worse, so only a race "
    "on that car can say what the lag costs."
)

for lags, worst in worst_error.items():
    print(f"car lag {lags[0]}, model lag {lags[1]}: largest |e| after the first second {worst:.4f} m")
print(toy_versus_car)

# independent check: the lesson's toy loop, with one variable for the command in flight, is (1, 1) and (1, 0)
for knows_lag in (True, False):
    e, psi, distance, acting, lesson_errors = 0.8, 0.0, 0.0, 0.0, []
    for _ in range(140):
        kappa = np.array([toy_kappa(distance + 18.0 * DT * (t + 1)) for t in range(40)])
        sent_now = mpc_tick(e, psi, np.zeros(40) + DT * 18.0, kappa, [acting] if knows_lag else [], acting)
        e, psi = nonlinear_step(e, psi, acting, DT * 18.0, toy_kappa(distance + 18.0 * DT))
        distance += 18.0 * DT
        acting = sent_now
        lesson_errors.append(e)
    assert np.max(np.abs(np.array(lesson_errors) - drives[(1, 1 if knows_lag else 0)])) == 0.0
assert abs(worst_error[(1, 1)] - 0.135) < 0.001 and abs(worst_error[(1, 0)] - 0.116) < 0.001  # the lesson's toy
assert max(worst_error.values()) < 0.2 and abs(drives[(2, 2)][0] - drives[(2, 0)][0]) < 1e-12
assert max(worst_error[(2, 0)], worst_error[(2, 1)], worst_error[(2, 2)]) - min(
    worst_error[(2, 0)], worst_error[(2, 1)], worst_error[(2, 2)]) < 0.05

## Exercise 5 - The tick budget

Run this exercise **on its own**, with no other notebook or program racing: a timing taken while the machine is busy measures the machine.

**The solve, race-free.**
Build an agent on `test-circuit` (`with shim_installed(): Unit10, module_name = load_submission(UNIT_10)`, then `agent = Unit10(load_track("test-circuit").waypoints, None)`), and take `start` = the index of the largest |`agent.curvature`| minus 20.
For each horizon m = 10, 20, 40 and 80: `v_ref, k_ref = horizon_reference(agent, start, agent.profile[start], m)`, then call `mpc_tick(0.8, 0.05, DT * v_ref, k_ref, [0.0], 0.0)` 220 times, timing each call with `time.perf_counter()`.
Drop the first 20 calls (the warm-up).

**The step, in a race.**
Race the shipped agent with no timing, `variant(UNIT_10, "shipped", **NO_TIMING)`, on `test-circuit`, seed 0, `max_seconds=450.0`; `result.step_ms` holds the time of every `step()` call (ms).

Define:

- `solve_p50`, `solve_p99`: dicts from m to the median and the 99th percentile (`np.percentile`) of the kept times (ms);
- `shipped_race`: the race result (Exercises 6 and 8 reuse it), and `shipped_offsets`: `np.array(agent.offsets)`, the shipped line's offsets on `test-circuit`;
- `shipped_clean`: `clean(shipped_race, shipped_offsets)`;
- `step_p50`, `step_p99`: the median and 99th percentile of `step_ms` after the first 50 steps (`check_submission`'s warm-up);
- `warmup_max`: the largest of the first 50;
- `inside_budget`: `True` when `step_p99` is at most 5.0 ms;
- `fig`, `ax`: `solve_p99` against m, with axis labels and a title;
- `why_not_in_a_pool`: two sentences: why can a timing taken while eight races share the machine be wrong when the same races' **times** are right?

**Prediction.** The least-squares problem has 3m rows and m columns, so the solve should grow faster than linearly with m: a fraction of a millisecond at m = 10–40 and over a millisecond at 80.
The lesson's probes measured the whole step at p50 about 0.9 ms and p99 about 1.0 ms on an idle machine, so `inside_budget` should be `True` with room to spare; the first 50 steps include the warm-up and can be slower.
The shipped agent at 0.58 on `test-circuit` should be clean, and its time should sit between the lesson's 0.60 race (272.40 s) and its 0.54 race (279.65 s), since a larger fraction plans a faster profile.

In [ ]:
with shim_installed():
    Unit10, module_name = load_submission(UNIT_10)
agent = Unit10(load_track("test-circuit").waypoints, None)
start = int(np.argmax(np.abs(agent.curvature))) - 20
solve_p50, solve_p99 = {}, {}
for m in (10, 20, 40, 80):
    v_ref, k_ref = horizon_reference(agent, start, agent.profile[start], m)
    times_ms = []
    for _ in range(220):
        started = time.perf_counter()
        mpc_tick(0.8, 0.05, DT * v_ref, k_ref, [0.0], 0.0)
        times_ms.append((time.perf_counter() - started) * 1000)
    solve_p50[m] = float(np.percentile(times_ms[20:], 50))
    solve_p99[m] = float(np.percentile(times_ms[20:], 99))

shipped_race = run_race(variant(UNIT_10, "shipped", **NO_TIMING), "test-circuit", car=carla_like(noise=True), seed=0,
                        max_seconds=450.0)
shipped_offsets = np.array(agent.offsets)
shipped_clean = clean(shipped_race, shipped_offsets)
step_p50 = float(np.percentile(shipped_race.step_ms[50:], 50))
step_p99 = float(np.percentile(shipped_race.step_ms[50:], 99))
warmup_max = float(np.max(shipped_race.step_ms[:50]))
inside_budget = step_p99 <= 5.0

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(list(solve_p99), list(solve_p99.values()), marker="o")
ax.set_xlabel("horizon m (ticks)")
ax.set_ylabel("p99 of one MPC tick (ms)")
ax.set_title("The solve's cost grows with the horizon (this machine)")
plt.show()
why_not_in_a_pool = (
    "A race's time is counted in the simulator's ticks, so it does not care how long each tick took on the wall "
    "clock, and eight races sharing the machine still finish in the same simulated time. "
    "A timing measures the wall clock, and with eight processes sharing the cores any step can be paused for "
    "milliseconds while another process runs, so the p99 measures the machine's crowding, not the agent."
)

for m in solve_p50:
    print(f"horizon {m:2d}: solve p50 {solve_p50[m]:.3f} ms, p99 {solve_p99[m]:.3f} ms")
print(f"shipped race: {shipped_race.total_time:.2f} s, clean {shipped_clean}, "
      f"{line_error(shipped_race, shipped_offsets):.2f} m from its line")
print(f"{shipped_race.step_ms.size} steps: p50 {step_p50:.3f} ms, p99 {step_p99:.3f} ms after the warm-up; "
      f"slowest of the first 50 {warmup_max:.2f} ms; inside the 5 ms budget: {inside_budget}")
print(why_not_in_a_pool)

# independent checks: the race against the lesson's bracketing races at 0.60 and 0.54 on test-circuit, one step()
# per tick of the race; the timings only by their order
assert shipped_clean and 272.40 - TICK <= shipped_race.total_time <= 279.65 + TICK
assert shipped_race.step_ms.size == round(shipped_race.total_time / DT)
assert solve_p50[80] > solve_p50[10] and step_p50 <= step_p99

## Exercise 6 - Your MPC inside the agent

The shipped agent calls its method `mpc(self, yaw, speed)` once a tick and does everything else itself: it clips the first command, applies it, queues it for the lag and keeps it as `mpc_last`.
So a subclass that **replaces** `mpc` with your own code races your MPC with everything else unchanged.

Write `agent_plan(agent, yaw, speed, n, weights=WEIGHTS)`, the plan the shipped `mpc()` computes, from the agent's attributes only:

1. the segment i = `agent.segment_index` and j = (i + 1) mod the number of points; the segment's heading is `math.atan2(agent.ys[j] - agent.ys[i], agent.xs[j] - agent.xs[i])`;
2. e = `agent.cross_track_error`, and ψ = yaw − heading, wrapped to [−π, π) (`(angle + math.pi) % (2 * math.pi) - math.pi`);
3. `v_ref, k_ref = horizon_reference(agent, i, speed, n)` and a = Δt·v_ref;
4. roll `linear_step` through the queued commands `agent.mpc_queue` (tick t uses a[t] and k_ref[t]);
5. `stack_horizon` on the rest of the horizon (a and k_ref from index `len(agent.mpc_queue)` on), and return `least_squares_plan(G, h, agent.mpc_last, weights)`.

Then load the class of Exercise 5's variant, `with shim_installed(): Shipped, _ = load_submission(variant(UNIT_10, "shipped", **NO_TIMING))`, and subclass it:

```python
class MyMPC(Shipped):
    def mpc(self, yaw, speed):
        return agent_plan(self, yaw, speed, self.horizon)
```

and race it on `test-circuit`, seed 0, `max_seconds=450.0` (`run_race` accepts a class).

Define:

- `agent_plan`: the function, and `MyMPC`: the class;
- `my_race`: the race result;
- `same_time`: `True` when `my_race.total_time` equals `shipped_race.total_time`;
- `largest_offset_gap`: the largest |difference| of the two races' telemetry `offset` columns (m), or `None` if they have different numbers of rows;
- `what_this_checks`: one sentence: what does an identical race tell you that a race-free test of `stack_horizon` does not?

**Prediction.** `agent_plan` is the same arithmetic as the shipped `mpc()`, in the same order, on the same attributes, so every tick's plan should agree to the last bit, and a race is deterministic for a seed: the same time to the tick and an offset gap of exactly 0.
If any step differed (the heading's sign, the wrap, the queue, the reference's half step), the race would drift apart somewhere in 5500 ticks.

In [ ]:
def agent_horizon(agent, yaw, speed, n):
    # Steps 1-5 up to the stack: (G, h) of the horizon after the queued commands, from the agent's attributes.
    count = len(agent.xs)
    i = agent.segment_index
    j = (i + 1) % count
    heading = math.atan2(agent.ys[j] - agent.ys[i], agent.xs[j] - agent.xs[i])
    e = agent.cross_track_error
    psi = (yaw - heading + math.pi) % (2 * math.pi) - math.pi
    v_ref, k_ref = horizon_reference(agent, i, speed, n)
    a = DT * v_ref
    lag = len(agent.mpc_queue)
    for t in range(lag):
        e, psi = linear_step(e, psi, agent.mpc_queue[t], a[t], k_ref[t])
    return stack_horizon(e, psi, a[lag:], k_ref[lag:])


def agent_plan(agent, yaw, speed, n, weights=WEIGHTS):
    G, h = agent_horizon(agent, yaw, speed, n)
    return least_squares_plan(G, h, agent.mpc_last, weights)


with shim_installed():
    Shipped, _ = load_submission(variant(UNIT_10, "shipped", **NO_TIMING))


class MyMPC(Shipped):
    def mpc(self, yaw, speed):
        return agent_plan(self, yaw, speed, self.horizon)


my_race = run_race(MyMPC, "test-circuit", car=carla_like(noise=True), seed=0, max_seconds=450.0)
same_time = my_race.total_time == shipped_race.total_time
largest_offset_gap = None
if len(my_race.telemetry["offset"]) == len(shipped_race.telemetry["offset"]):
    largest_offset_gap = float(np.max(np.abs(my_race.telemetry["offset"] - shipped_race.telemetry["offset"])))
what_this_checks = (
    "A race-free test checks that stack_horizon builds the right matrices from given numbers, but an identical race "
    "checks everything around it as well: that the errors are read from the right attributes with the right signs "
    "and wrap, that the reference and the lag queue line up with the plan, and that every one of thousands of ticks "
    "of the closed loop gives the shipped agent's command."
)

print(f"my MPC: {my_race.total_time:.2f} s, clean {clean(my_race, shipped_offsets)}; "
      f"shipped: {shipped_race.total_time:.2f} s")
print(f"same time: {same_time}; largest offset gap: {largest_offset_gap}")
print(what_this_checks)

# independent check: the shipped race was raced by racekit from the file; this one ran my code in its place
assert same_time and largest_offset_gap == 0.0
assert len(my_race.telemetry["t"]) == len(shipped_race.telemetry["t"]) and clean(my_race, shipped_offsets)

## Exercise 7 - The honest comparison on `test-circuit-seed1`

The lesson found that at the same f, on `test-circuit`, MPC is no faster than pure pursuit, and that its gain is the tracking margin.
Is that a `test-circuit` accident?
Solve `test-circuit-seed1`'s line (`seed1 = load_track("test-circuit-seed1")`, `seed1_control = solve_line(seed1)`) and race both agents at f = 0.54 with that track's line and measured grip:

- MPC: `on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-mpc", MU_FRACTION=0.54, **NO_TIMING)`;
- pure pursuit: `on_track(UNIT_09, "test-circuit-seed1", seed1_control, "seed1-pursuit")` (Unit 9 ships 0.54);

each with `repeat_races` on `"test-circuit-seed1"`, seeds 0, 1 and 2 (`n=3, seeds=(0, 1, 2)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.

Define:

- `seed1_offsets`: `spline_basis(len(seed1)) @ seed1_control`;
- `mpc_results`, `pursuit_results`: the two `RepeatResult` objects;
- `all_clean`: `True` when all six races are clean (against `seed1_offsets`);
- `medians`, `spreads`: dicts `{"mpc": ..., "pursuit": ...}` (s);
- `worst_errors`: a dict `{"mpc": ..., "pursuit": ...}` of the largest `line_error` over each agent's three races (m);
- `mpc_better`: `noise_aware_better(mpc_results.times, pursuit_results.times)`, and `pursuit_better`: the other way round;
- `margin_m`: `worst_errors["pursuit"] - worst_errors["mpc"]`;
- `verdict`: two sentences: does this track tell the same story as the lesson's `test-circuit`? What would you spend the margin on, and by which rule?

**Prediction.** Both agents drive the same line with the same speed profile at the same f, and the lesson found that the profile, not the steering, sets the lap time; so I expect the two medians within a tick or two of each other, neither noise-aware better.
The lesson's table puts pursuit at 0.54 about 1.7 m from its line at worst and MPC about 1.2 m, so all six races should be clean, with a margin of a few tenths of a metre for MPC.

In [ ]:
seed1 = load_track("test-circuit-seed1")
seed1_control = solve_line(seed1)
seed1_offsets = spline_basis(len(seed1)) @ seed1_control
mpc_results = repeat_races(
    on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-mpc", MU_FRACTION=0.54, **NO_TIMING),
    "test-circuit-seed1", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
pursuit_results = repeat_races(
    on_track(UNIT_09, "test-circuit-seed1", seed1_control, "seed1-pursuit"),
    "test-circuit-seed1", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
all_clean = False not in [clean(result, seed1_offsets) for result in mpc_results.results + pursuit_results.results]
times = {"mpc": [result.total_time for result in mpc_results.results],
         "pursuit": [result.total_time for result in pursuit_results.results]}
medians = {name: float(np.median(values)) for name, values in times.items()}
spreads = {name: max(values) - min(values) for name, values in times.items()}
worst_errors = {"mpc": max(line_error(result, seed1_offsets) for result in mpc_results.results),
                "pursuit": max(line_error(result, seed1_offsets) for result in pursuit_results.results)}
mpc_better = noise_aware_better(mpc_results.times, pursuit_results.times)
pursuit_better = noise_aware_better(pursuit_results.times, mpc_results.times)
margin_m = worst_errors["pursuit"] - worst_errors["mpc"]
verdict = (
    f"Yes: at the same f the medians are {medians['mpc']:.2f} s (MPC) and {medians['pursuit']:.2f} s (pursuit), "
    f"neither noise-aware better, while MPC stays {worst_errors['mpc']:.2f} m from its line at worst against "
    f"pursuit's {worst_errors['pursuit']:.2f} m, so the gain is again the tracking margin ({margin_m:.2f} m), not "
    "speed. "
    "I would spend the margin on a higher MU_FRACTION by the margin rule: raise f a grid step at a time on every "
    "training track and several seeds until a race is not clean, freeze one step inside that family edge, and only "
    "then check the transfer on a track never tuned on."
)

for name, results in (("mpc", mpc_results), ("pursuit", pursuit_results)):
    print(f"{name:8}: {[round(float(t), 2) for t in results.times]} s, "
          f"{[round(line_error(result, seed1_offsets), 2) for result in results.results]} m from the line")
print(f"all clean: {all_clean}; medians: MPC {medians['mpc']:.2f} s, pursuit {medians['pursuit']:.2f} s; "
      f"spreads: MPC {spreads['mpc']:.2f} s, pursuit {spreads['pursuit']:.2f} s")
print(f"MPC better: {mpc_better}; pursuit better: {pursuit_better}; margin {margin_m:.2f} m")
print(verdict)

# independent checks: the RepeatResults' own medians and spreads, Unit 1's rule by hand, and the lesson's story
assert medians["mpc"] == mpc_results.median and medians["pursuit"] == pursuit_results.median
assert abs(spreads["mpc"] - mpc_results.spread) < 1e-9 and abs(spreads["pursuit"] - pursuit_results.spread) < 1e-9
noise = max(spreads.values())
assert mpc_better == (medians["pursuit"] - medians["mpc"] > noise)
assert pursuit_better == (medians["mpc"] - medians["pursuit"] > noise)
assert all_clean and not mpc_better and not pursuit_better
assert abs(medians["mpc"] - medians["pursuit"]) <= 2 * TICK and margin_m > 0.1

## Exercise 8 - How long a horizon?

The shipped agent plans 40 ticks (2 s).
The lesson's §7 showed the solve's cost growing with the horizon; what does a shorter horizon cost on the track?
Race the shipped agent with `MPC_HORIZON = 10` and with `MPC_HORIZON = 20` (each `variant(UNIT_10, ..., MPC_HORIZON=..., **NO_TIMING)`) on `test-circuit`, seed 0, `max_seconds=450.0`.
The 40-tick race is Exercise 5's `shipped_race`.

Define:

- `horizon_races`: a dict from the horizon (10, 20, 40) to its race result;
- `horizon_clean`: a dict from the horizon to `clean(result, shipped_offsets)`;
- `horizon_times`: a dict from the horizon to the race time (s), and `horizon_errors`: to its `line_error` against `shipped_offsets` (m);
- `cost_of_10`: `horizon_times[10] - horizon_times[40]` (s);
- `fig`, `ax`: the race time against Exercise 5's `solve_p50` for the three horizons, one point per horizon, each plotted with its own `label` (for example `"N = 10"`) so that the legend names it, with axis labels and a title;
- `which_horizon`: two sentences: if your machine's budget for the solve were a p99 of 0.5 ms, which horizon would you ship, and what evidence would you still want before trusting it (one seed, one track)?

**Prediction.** The lesson's probe at 0.60 on `test-circuit` found 273.05 s at N = 10, 272.45 s at 20 and 272.40 s at 40: a shorter horizon costs a little time, most of it between 10 and 20.
At the shipped 0.58 I expect the same shape: all three clean, N = 10 slower by about half a second, N = 20 within a tick or two of N = 40.
A shorter horizon sees less of the next corner, so it should also follow the line a little less closely.

In [ ]:
horizon_races = {40: shipped_race}
for N in (10, 20):
    horizon_races[N] = run_race(variant(UNIT_10, f"horizon-{N}", MPC_HORIZON=N, **NO_TIMING), "test-circuit",
                                car=carla_like(noise=True), seed=0, max_seconds=450.0)
horizon_races = {N: horizon_races[N] for N in (10, 20, 40)}
horizon_clean = {N: clean(result, shipped_offsets) for N, result in horizon_races.items()}
horizon_times = {N: result.total_time for N, result in horizon_races.items()}
horizon_errors = {N: line_error(result, shipped_offsets) for N, result in horizon_races.items()}
cost_of_10 = horizon_times[10] - horizon_times[40]

fig, ax = plt.subplots(figsize=(7, 3.5))
for N in (10, 20, 40):
    ax.scatter(solve_p50[N], horizon_times[N], label=f"N = {N}")
ax.set_xlabel("p50 of one MPC tick at that horizon (ms, Exercise 5)")
ax.set_ylabel("race time (s)")
ax.set_title("test-circuit, f = 0.58, seed 0: time on the track against time per solve")
ax.legend()
plt.show()
fits = [N for N in (10, 20, 40) if solve_p99[N] <= 0.5]
if fits:
    choice = (f"With a 0.5 ms budget for the solve I would ship N = {max(fits)}, the longest horizon whose p99 here "
              f"({solve_p99[max(fits)]:.2f} ms) fits, which costs {horizon_times[max(fits)] - horizon_times[40]:.2f} s "
              "against N = 40 on this race. ")
else:
    choice = ("With a 0.5 ms budget none of these horizons fits on this machine right now, so I would keep the "
              "accepted configuration and measure again on an idle machine. ")
which_horizon = choice + (
    "Before trusting it I would race it on every training track and several seeds, and on a track never tuned on, "
    "clean each time, and time check_submission alone on the machine it will race on: one seed on one track says "
    "nothing about the spread."
)

for N in (10, 20, 40):
    print(f"N = {N:2d}: {horizon_times[N]:.2f} s, clean {horizon_clean[N]}, {horizon_errors[N]:.2f} m from the line, "
          f"solve p50 {solve_p50[N]:.3f} ms")
print(f"cost of N = 10: {cost_of_10:.2f} s")
print(which_horizon)

# independent checks: the 40-tick race is Exercise 5's; the lesson's shape (Section 7: a shorter horizon costs
# about half a second, not the race)
assert horizon_races[40] is shipped_race and horizon_times[40] == shipped_race.total_time
assert False not in list(horizon_clean.values())
assert horizon_times[10] >= horizon_times[20] >= horizon_times[40] and 0.0 < cost_of_10 < 2.0
assert horizon_errors[10] > horizon_errors[40]

## Exercise 9 - Challenge: a terminal cost

The horizon cost weighs every tick's error the same.
A common variation pays **more** for the error at the end of the horizon, so that the plan must end close to the line: a **terminal cost**.
Multiply the last state's two weights, q_e and q_ψ at t = m, by a factor `terminal` (in the stacked rows: multiply the last two entries of W's diagonal by √terminal).

Write:

- `terminal_plan(G, h, u_last, terminal, weights=WEIGHTS)`: Exercise 3's `least_squares_plan` with that change (with `terminal = 1.0` it must give the same plan);
- `agent_terminal_plan(agent, yaw, speed, n, terminal, weights=WEIGHTS)`: Exercise 6's `agent_plan`, solving with `terminal_plan`.

The probe record of plan `roar-009` found the shipped MPC's family edge at 0.60: at 0.62 on `test-circuit-seed1` it was **not clean on seeds 0, 3 and 4** (clean on seeds 1 and 2).
Does a terminal cost move that edge?
Make a subclass, as in Exercise 6, of the class loaded from `on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-0.62", MU_FRACTION=0.62, **NO_TIMING)`, whose `mpc` returns `agent_terminal_plan(self, yaw, speed, self.horizon, 10.0)`, and race it on `"test-circuit-seed1"`, seeds 0 and 3, `max_seconds=450.0`.

Define:

- `terminal_plan`, `agent_terminal_plan`: the functions;
- `same_plan_at_1`: `True` when `terminal_plan(G, h, 0.0, 1.0)` equals `least_squares_plan(G, h, 0.0)` within 1e-12 on Exercise 2's test horizon;
- `end_error`: a dict from `terminal` (1.0, 10.0, 100.0) to the planned |e_30| on Exercise 2's test horizon (from (0.6, −0.04), u_last = 0);
- `terminal_races`: a dict from seed (0, 3) to the race result, and `terminal_clean`: a dict from seed to `clean(result, seed1_offsets)`;
- `terminal_errors`: a dict from seed to `line_error(result, seed1_offsets)` (m);
- `moved_the_edge`: two or three sentences: on these two seeds, is the terminal cost clean where the shipped MPC was not? What more would you have to race before you raised `MU_FRACTION` (the margin rule, the family, the transfer)?

**Prediction.** On the test horizon a larger terminal factor pays more for the last state, so the planned |e_30| should fall as the factor grows; with factor 1 the stacked rows are unchanged, so the plan is the same.
On the car I doubt it moves the edge.
The races at 0.62 fail where the planned **speed** asks for more grip than the car has in a corner; a terminal cost changes only where the plan wants to be in 2 s, and the first command, the only one applied, barely changes.
I expect the two races to stay not clean.

In [ ]:
def terminal_plan(G, h, u_last, terminal, weights=WEIGHTS):
    q_e, q_psi, r = weights
    m = G.shape[1]
    w = np.tile([math.sqrt(q_e), math.sqrt(q_psi)], m)
    w[-2:] = w[-2:] * math.sqrt(terminal)  # the last state's two rows: q_e and q_psi times terminal
    D = np.eye(m) - np.eye(m, k=-1)
    d = np.zeros(m)
    d[0] = u_last
    rows = np.vstack([w[:, None] * G, math.sqrt(r) * D])
    rhs = np.concatenate([-w * h, math.sqrt(r) * d])
    return np.linalg.lstsq(rows, rhs, rcond=None)[0]


def agent_terminal_plan(agent, yaw, speed, n, terminal, weights=WEIGHTS):
    G, h = agent_horizon(agent, yaw, speed, n)
    return terminal_plan(G, h, agent.mpc_last, terminal, weights)


same_plan_at_1 = float(np.max(np.abs(terminal_plan(G_test, h_test, 0.0, 1.0)
                                     - least_squares_plan(G_test, h_test, 0.0)))) <= 1e-12
end_error = {}
for terminal in (1.0, 10.0, 100.0):
    end_error[terminal] = abs((G_test @ terminal_plan(G_test, h_test, 0.0, terminal) + h_test)[-2])

with shim_installed():
    Seed1At062, _ = load_submission(on_track(UNIT_10, "test-circuit-seed1", seed1_control, "seed1-0.62",
                                             MU_FRACTION=0.62, **NO_TIMING))


class TerminalMPC(Seed1At062):
    def mpc(self, yaw, speed):
        return agent_terminal_plan(self, yaw, speed, self.horizon, 10.0)


terminal_races, terminal_clean, terminal_errors = {}, {}, {}
for seed in (0, 3):
    terminal_races[seed] = run_race(TerminalMPC, "test-circuit-seed1", car=carla_like(noise=True), seed=seed,
                                    max_seconds=450.0)
    terminal_clean[seed] = clean(terminal_races[seed], seed1_offsets)
    terminal_errors[seed] = line_error(terminal_races[seed], seed1_offsets)
moved_the_edge = (
    "No: with a terminal factor of 10 the MPC at 0.62 is still not clean on seeds 0 and 3 "
    f"({terminal_races[0].collisions} and {terminal_races[3].collisions} collisions, "
    f"{terminal_errors[0]:.2f} and {terminal_errors[3]:.2f} m from its line, "
    f"finished: {terminal_races[0].finished} and {terminal_races[3].finished}), "
    "so on these seeds it does not move the edge: it changes where each plan ends in 2 s, not the speed the profile "
    "asks of the tyres. "
    "Before raising MU_FRACTION I would need it clean on every seed of every training track at the new value and "
    "at least one grid step above it (the family and the margin rule), with an honest comparison against the "
    "shipped weights, and then clean on a track never tuned on (the transfer)."
)

print(f"same plan at terminal = 1: {same_plan_at_1}")
for terminal, error in end_error.items():
    print(f"terminal {terminal:5.1f}: planned |e_30| {error:.4f} m")
for seed, result in terminal_races.items():
    print(f"seed {seed}: {result.total_time:.2f} s, {result.collisions} collisions, finished {result.finished}, "
          f"{terminal_errors[seed]:.2f} m from its line, clean {terminal_clean[seed]}")
print(moved_the_edge)

# independent checks: Exercise 3's plan and end error are the factor-1 case; a bigger factor ends nearer the line
assert same_plan_at_1 and abs(end_error[1.0] - final_error[1000.0]) < 1e-12
assert end_error[1.0] > end_error[10.0] > end_error[100.0]
assert terminal_clean == {0: False, 3: False}

## Exercise 10 - Challenge: plan a distance, not a time

A 40-tick horizon looks 2 s ahead: 90 m at 45 m/s, but only 20 m at 10 m/s, in the slowest corners, where the line turns most.
Try a horizon that looks a fixed **distance** ahead instead: at speed v,

N(v) = round(DISTANCE / (Δt · max(v, 1))), clipped to 10 … 60 ticks, with DISTANCE = 40 m.

Make a subclass, as in Exercise 6, of the class loaded from `on_track(UNIT_10, "test-circuit-seed4", seed4_control, "seed4-distance", **NO_TIMING)` (`seed4_control = solve_line(load_track("test-circuit-seed4"))`; the shipped `MU_FRACTION`, 0.58), whose `mpc` returns `agent_plan(self, yaw, speed, N(speed))`.
Race it with `repeat_races` on `"test-circuit-seed4"`, seeds 0, 1 and 2, `car_factory=lambda: carla_like(noise=True)`, `max_seconds=450.0`.
The shipped agent's times on those seeds (probe record, plan `roar-009` U10.3) are `SHIPPED_SEED4 = [271.10, 271.10, 271.15]`.

Define:

- `distance_horizon(speed)`: N(v), an int;
- `horizons_seen`: a dict from speed (8.0, 15.0, 25.0, 45.0 m/s) to `distance_horizon(speed)`;
- `seed4_offsets`: the line's offsets, and `distance_results`: the `RepeatResult`;
- `distance_clean`: `True` when all three races are clean;
- `distance_median`: the median time (s), and `worst_line_error`: the largest `line_error` of the three (m);
- `distance_better`: `noise_aware_better(distance_results.times, SHIPPED_SEED4)`, and `shipped_better`: the other way round;
- `worth_it`: two or three sentences: is the distance horizon noise-aware better than the shipped 2 s horizon here? Its longest horizon (60 ticks) costs more time per solve than 40 (Exercise 5): would you ship it, and what else would you race first?

**Prediction.** By hand: N(8) = 40 / 0.4 = 100, clipped to 60; N(15) = 40 / 0.75 = 53.3, so 53; N(25) = 40 / 1.25 = 32; N(45) = 40 / 2.25 = 17.8, so 18.
So the distance horizon plans longer than 40 ticks below 20 m/s, in the slow corners, and shorter on the fast stretches.
Exercise 8 found the horizon's effect on the time small (a tick or a few between 20 and 40), so I expect races clean and within a tick of the shipped 271.10 s, not noise-aware better either way.

In [ ]:
SHIPPED_SEED4 = [271.10, 271.10, 271.15]  # s: agents/unit-10.py on test-circuit-seed4, seeds 0-2 (plan roar-009)
DISTANCE = 40.0  # m: how far ahead the horizon looks


def distance_horizon(speed):
    return int(min(max(round(DISTANCE / (DT * max(speed, 1.0))), 10), 60))


horizons_seen = {speed: distance_horizon(speed) for speed in (8.0, 15.0, 25.0, 45.0)}
seed4 = load_track("test-circuit-seed4")
seed4_control = solve_line(seed4)
seed4_offsets = spline_basis(len(seed4)) @ seed4_control

with shim_installed():
    Seed4, _ = load_submission(on_track(UNIT_10, "test-circuit-seed4", seed4_control, "seed4-distance", **NO_TIMING))


class DistanceMPC(Seed4):
    def mpc(self, yaw, speed):
        return agent_plan(self, yaw, speed, distance_horizon(speed))


distance_results = repeat_races(DistanceMPC, "test-circuit-seed4", n=3, seeds=(0, 1, 2),
                                car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
distance_clean = False not in [clean(result, seed4_offsets) for result in distance_results.results]
distance_median = distance_results.median
worst_line_error = max(line_error(result, seed4_offsets) for result in distance_results.results)
distance_better = noise_aware_better(distance_results.times, SHIPPED_SEED4)
shipped_better = noise_aware_better(SHIPPED_SEED4, distance_results.times)
worth_it = (
    f"No: the distance horizon is clean ({worst_line_error:.2f} m from its line at worst) but its times "
    f"{[round(float(t), 2) for t in distance_results.times]} s against the shipped agent's "
    f"{SHIPPED_SEED4} s are not "
    "noise-aware better, and neither is the shipped horizon. "
    f"I would not ship it: it plans up to 60 ticks in the slow corners, and Exercise 5's solve grows from "
    f"{solve_p50[40]:.2f} ms at 40 ticks to {solve_p50[80]:.2f} ms at 80, for nothing measurable here; before shipping any such change I would race it on every training track "
    "and on an unseen one, and time it alone with check_submission."
)

print(f"horizons seen: {horizons_seen}")
print(f"distance horizon: {[round(float(t), 2) for t in distance_results.times]} s, "
      f"{[round(line_error(result, seed4_offsets), 2) for result in distance_results.results]} m from the line")
print(f"clean {distance_clean}; median {distance_median:.2f} s against the shipped {np.median(SHIPPED_SEED4):.2f} s; "
      f"distance better {distance_better}, shipped better {shipped_better}")
print(worth_it)

# independent checks: N(v) by hand, the line against the track's own length, and Unit 1's rule by hand
assert horizons_seen == {8.0: 60, 15.0: 53, 25.0: 32, 45.0: 18}
assert len(seed4_offsets) == len(seed4) and np.max(np.abs(seed4_offsets)) <= np.max(seed4.half_width) - 2.5 + 1e-9
assert distance_clean and not distance_better and not shipped_better
assert abs(distance_median - np.median(SHIPPED_SEED4)) <= TICK